### How to view deep agents in agentvis visualizer

#### API Key Setup - In this case for gemini and tavily search

In [1]:
import os
from google.oauth2 import service_account

os.environ["TAVILY_API_KEY"] = "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"
credentials = service_account.Credentials.from_service_account_file("credentials.json",scopes=["https://www.googleapis.com/auth/cloud-platform"])
project_id = "healthpay-434611"
google_auth_kwargs = {"credentials": credentials, "project": project_id}

#### Define your llm model

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0.7,
    **google_auth_kwargs,
)

#### Define your Deep Agent with tools, subagent, sytem prompt etc.

In [3]:
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent
from collections import defaultdict

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

research_subagent = {
    "name": "research-agent",
    "description": "Used to research more in depth questions",
    "system_prompt": "You are a great researcher",
    "tools": [internet_search],
    "model": llm,
}
subagents = [research_subagent]

agent = create_deep_agent(model=llm,subagents=subagents)

#### Run you deep-agents and capture messages and subagent messages using langchain streaming

In [4]:
messages = []
subagent_messages = defaultdict(list)
namespace_to_tool_mapping = {}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "what are the main components used to make cough syrup and give me detailed answer for each component"}]},
    stream_mode=["updates", "tasks"],
    subgraphs=True,
):
    if not (isinstance(chunk, tuple) and len(chunk) == 3):
        continue
    namespace, event_type, payload = chunk
    is_subagent = bool(namespace)

    if event_type == "tasks" and isinstance(payload, dict):
        if payload.get("name") != "tools":
            continue
        task_id = payload.get("id", "")
        tool_call = (payload.get("input") or {}).get("tool_call")
        if not task_id or not isinstance(tool_call, dict):
            continue
        tool_call_id = tool_call.get("id")
        if not tool_call_id:
            continue
        namespace_to_tool_mapping[f"tools:{task_id}"] = tool_call_id
    elif event_type == "updates" and isinstance(payload, dict):
        if is_subagent:
            tool_call_id = next(s for s in namespace if s.startswith("tools:"))
            for _, data in payload.items():
                if data and "messages" in data:
                    if hasattr(data["messages"], "value"):
                        new_msgs = data["messages"].value
                    else:
                        new_msgs = data["messages"]
                    subagent_messages[tool_call_id].extend(new_msgs)
        else:
            for _, data in payload.items():
                if data and "messages" in data:
                    if hasattr(data["messages"], "value"):
                        new_msgs = data["messages"].value
                    else:
                        new_msgs = data["messages"]
                    messages.extend(new_msgs)

# this is to map the tool call id to the subagent message id
# so if let say subagent run and return tool message then this tool message id will be mapped to the subagent message id
old_keys = list(subagent_messages.keys())
for key in old_keys:
    new_key = namespace_to_tool_mapping[key]
    subagent_messages[new_key] = subagent_messages.pop(key)

#### Use agentvis tool to visualize agent run in more detail

In [5]:
from agentvis.framework.langchain import LangChainAdapter
from agentvis.core.export import ExportFactory
from agentvis.core import AgentVis

graph = AgentVis.build_agent_graph(messages=LangChainAdapter().convert(messages,subagent_messages))
link = ExportFactory.export_graph(graph=graph, export_strategy="link")  # export_strategy = "link" | "json"
link

/Users/niteshkumar/Documents/data-extraction/.agentvisenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'https://agentvis.vercel.app?view=eJztfXtz20ay71dBeeusJV2CAkEQBJ26lZIlO9bZta0TyZvkJLmuATAksQYBBgAlM7v57rcfM8AABPWyvZtTJ39YJkFgMNPT09P968f848m8ECtZPnn24z-eZHmsPiXxk2dPRpHw59E8sENPjm1vHE5sEU3nduTNpm4Qymgi5JPBk2q7lnD3q81KZK9lWYoFXo1FJZ48-8eTKM8qmVVww81SVJYopFUtpbUSSWZF-WqdZ_BraW1KGVtVDtc_SLi-WSytclts1pbIYmuRXMMT0oplJZIUbhRZeSMLa54XlhTRsmnoyW-__fzbYGckafS-2GS27YxmkYxEZDuRM7Gn3si1het79mQyCWTgyliGru00Yzo5v21Ae14WuTMvmDgj252NJrY3iV07mDmuHTtyNAqnMo79cfOKqzxPb3vJ2zmSoyjlwDq3IpFZS5murZukWgIdRTW0Xkkg6T6y5vMWMekx0dBRflynIhNVkmc1MZ_9lP2U_elPf7JGQ-tMfqyKfCWrZV6sl_Dyg7PvXx_iDUdH_b8dHVlJCa9Qb92s1wUOLoOOnlfWTV58KK1wa4n5XEZVki2o12WyyERaWtB3_BoWgj4Bu1RFsljATONlbrKQ81R-hK59_5rfVG4z-LVKooGFndmmooKRxdS7Iq9EAW-D1vPFRiI5UnnN_c3TAT69CctKZJGEhvlJYMIIJjTJJPGesLI8s_N1kifYapFcw13Y0AqbgbtoYEnDwZUE6heiSNIttpnIa931SNA9MHwezmqV42Sl8YAuzNPNwIJJyOFLAdezOMGJKfULYErxvTTNeVnpBgTQEV4BtCxknNCsA_Xya1nYcKsNrwZWKlQXVnBLBJ0uh3qW3aH1zUYkcwkXk4yn1rig5jNDVoEZg4F1p7ICImQ4k0itNM9LSd9Wm2hTz6hIihuxLQdWlEqgDPwMw1vIEsc3oAdh3ePlpAIeLBOc8V